In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# configurations
pd.set_option('display.max_columns', None)

In [4]:
df = pd.read_csv("../data/processed/cleaned_data.csv")

In [5]:
df.columns

Index(['Gender', 'Age', 'Driving_License', 'Region_Code', 'Previously_Insured',
       'Vehicle_Age', 'Vehicle_Damage', 'Annual_Premium',
       'Policy_Sales_Channel', 'Vintage', 'Churn'],
      dtype='object')

#  1. CUSTOMER TYPE

In [6]:
df['Is_New_Customer'] = df['Previously_Insured'].apply(lambda x: 1 if x == 0 else 0)

# 2. RISK FEATURES

In [7]:
df['Has_Damage'] = df['Vehicle_Damage'].apply(lambda x: 1 if x == 'Yes' else 0)
df['High_Risk_Customer'] = df['Has_Damage'] * df['Is_New_Customer']

# 3. VEHICLE AGE (ORDINAL)


In [8]:
vehicle_map = {
    '< 1 Year': 0,
    '1-2 Year': 1,
    '> 2 Years': 2
}
df['Vehicle_Age_Num'] = df['Vehicle_Age'].map(vehicle_map)

# 4. AGE FEATURES

In [9]:
df['Age_Group_Risk'] = df['Age'].apply(lambda x: 1 if 30 <= x <= 50 else 0)


# 5. PREMIUM FEATURES

In [10]:
df['Premium_Log'] = np.log1p(df['Annual_Premium'])

df['Premium_Per_Day'] = df['Annual_Premium'] / (df['Vintage'] + 1)

# 6. INTERACTION FEATURES

In [11]:
df['Age_Vehicle'] = df['Age'] * df['Vehicle_Age_Num']

df['Damage_Vehicle'] = df['Has_Damage'] * df['Vehicle_Age_Num']

# 7. CHANNEL ENCODING (IMPORTANT)


In [12]:
channel_counts = df['Policy_Sales_Channel'].value_counts()
df['Channel_Frequency'] = df['Policy_Sales_Channel'].map(channel_counts)

# 8. REGION ENCODING

In [13]:
region_counts = df['Region_Code'].value_counts()
df['Region_Frequency'] = df['Region_Code'].map(region_counts)

# 9. CUSTOMER VALUE SCORE

In [14]:
df['Customer_Value'] = df['Premium_Per_Day'] * df['Vintage']

# 10. RISK SCORE (Composite)

In [15]:
df['Risk_Score'] = (
    df['Has_Damage'] * 2 +
    df['Vehicle_Age_Num'] +
    df['Age_Group_Risk']
)

# 11. ENGAGEMENT PROXY

In [16]:
df['Engagement_Score'] = (
    (1 - df['Is_New_Customer']) +
    df['Vintage'] / df['Vintage'].max()
)

In [17]:
df = df.drop(columns=[
    'Vehicle_Age',
    'Vehicle_Damage'
])

In [18]:
display(df.head())
print(df.shape)
print(df.describe())

,Gender,Age,Driving_License,Region_Code,Previously_Insured,Annual_Premium,Policy_Sales_Channel,Vintage,Churn,Is_New_Customer,Has_Damage,High_Risk_Customer,Vehicle_Age_Num,Age_Group_Risk,Premium_Log,Premium_Per_Day,Age_Vehicle,Damage_Vehicle,Channel_Frequency,Region_Frequency,Customer_Value,Risk_Score,Engagement_Score
0,Male,44,1,28.0,0,40454.0,26.0,217,0,1,1,1,2,1,10.607946,185.568807,88,2,79700,106415,40268.431193,5,0.725753
1,Male,76,1,3.0,0,33536.0,26.0,183,1,1,0,0,1,0,10.420405,182.260870,76,0,79700,9251,33353.739130,1,0.612040
2,Male,47,1,28.0,0,38294.0,26.0,27,0,1,1,1,2,1,10.553075,1367.642857,94,2,79700,106415,36926.357143,5,0.090301
3,Male,21,1,11.0,1,28619.0,152.0,203,1,0,0,0,0,0,10.261861,140.289216,0,0,134784,9232,28478.710784,0,1.678930
4,Female,29,1,41.0,1,27496.0,152.0,39,1,0,0,0,0,0,10.221832,687.400000,0,0,134784,18263,26808.600000,0,1.130435


(381109, 23)
                 Age  Driving_License    Region_Code  Previously_Insured  \
count  381109.000000    381109.000000  381109.000000       381109.000000   
mean       38.822584         0.997869      26.388807            0.458210   
std        15.511611         0.046110      13.229888            0.498251   
min        20.000000         0.000000       0.000000            0.000000   
25%        25.000000         1.000000      15.000000            0.000000   
50%        36.000000         1.000000      28.000000            0.000000   
75%        49.000000         1.000000      35.000000            1.000000   
max        85.000000         1.000000      52.000000            1.000000   

       Annual_Premium  Policy_Sales_Channel        Vintage          Churn  \
count   381109.000000         381109.000000  381109.000000  381109.000000   
mean     30338.717275            112.034295     154.347397       0.877437   
std      15916.679308             54.203995      83.671304       0.3279

In [19]:
df_model = df.drop(columns=[
    'Driving_License',
    'Policy_Sales_Channel',
    'Region_Code',
    'Customer_Value',
    'Engagement_Score'
])

In [20]:
df_model

,Gender,Age,Previously_Insured,Annual_Premium,Vintage,Churn,Is_New_Customer,Has_Damage,High_Risk_Customer,Vehicle_Age_Num,Age_Group_Risk,Premium_Log,Premium_Per_Day,Age_Vehicle,Damage_Vehicle,Channel_Frequency,Region_Frequency,Risk_Score
0,Male,44,0,40454.0,217,0,1,1,1,2,1,10.607946,185.568807,88,2,79700,106415,5
1,Male,76,0,33536.0,183,1,1,0,0,1,0,10.420405,182.260870,76,0,79700,9251,1
2,Male,47,0,38294.0,27,0,1,1,1,2,1,10.553075,1367.642857,94,2,79700,106415,5
3,Male,21,1,28619.0,203,1,0,0,0,0,0,10.261861,140.289216,0,0,134784,9232,0
4,Female,29,1,27496.0,39,1,0,0,0,0,0,10.221832,687.400000,0,0,134784,18263,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
381104,Male,74,1,30170.0,88,1,0,0,0,1,0,10.314636,338.988764,74,0,79700,2587,1
381105,Male,30,1,40016.0,131,1,0,0,0,0,1,10.597060,303.151515,0,0,134784,5501,1
381106,Male,21,1,35118.0,161,1,0,0,0,0,0,10.466498,216.777778,0,0,21779,12191,0
381107,Female,68,0,44617.0,74,1,1,1,1,2,0,10.705893,594.893333,136,2,73995,4678,4


In [21]:
df_model.head(2)

,Gender,Age,Previously_Insured,Annual_Premium,Vintage,Churn,Is_New_Customer,Has_Damage,High_Risk_Customer,Vehicle_Age_Num,Age_Group_Risk,Premium_Log,Premium_Per_Day,Age_Vehicle,Damage_Vehicle,Channel_Frequency,Region_Frequency,Risk_Score
0,Male,44,0,40454.0,217,0,1,1,1,2,1,10.607946,185.568807,88,2,79700,106415,5
1,Male,76,0,33536.0,183,1,1,0,0,1,0,10.420405,182.260870,76,0,79700,9251,1


In [22]:
df.to_csv("../data/processed/featured_data.csv", index = False)